# Adaptive Cover Pro — Simulation Notebook

Simulate cover positions for **any date or date range** using your actual Home Assistant configuration.

## Quick Start
1. In Home Assistant: **Developer Tools → Services → Adaptive Cover Pro: Export Configuration** → select your cover → **Call Service** → copy the JSON response
2. Paste the JSON into **Cell 2** below
3. Set `START_DATE` and `NUM_DAYS` in **Cell 2**
4. Run all cells: **Cell → Run All**

## Plot Guide
- **Red dashed lines** — Sunrise and sunset
- **Yellow dashed lines** — Sun enters/exits the window's field of view
- **Shaded region** — Sun is actively in front of the window
- **Top subplot** — Sun elevation and azimuth angles
- **Bottom subplot** — Calculated cover position (%), gamma angle on secondary axis

In [ ]:
# Cell 1 — Imports
import sys
sys.path.append("../custom_components")

import json
from datetime import date, timedelta
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import numpy as np

from adaptive_cover_pro.calculation import (
    AdaptiveVerticalCover,
    AdaptiveHorizontalCover,
    AdaptiveTiltCover,
)
from adaptive_cover_pro.sun import SunData

%matplotlib inline

In [ ]:
# Cell 2 — CONFIGURATION
#
# Step 1: In Home Assistant go to Developer Tools → Services
#         Select "Adaptive Cover Pro: Export Configuration", pick your cover, click Call Service
#         Copy the entire JSON response and paste it between the triple-quotes below.
#
# Step 2: Set START_DATE and NUM_DAYS
#         START_DATE: "today" uses today's date, or use "YYYY-MM-DD" (e.g. "2026-06-21")
#         NUM_DAYS: number of consecutive days to simulate (default 1 = single 24-hour day)

CONFIG_JSON = """
{
  "export_version": 1,
  "name": "Example Cover",
  "cover_type": "cover_blind",
  "location": {
    "latitude": 32.939,
    "longitude": -117.156,
    "elevation": 0,
    "timezone": "America/Los_Angeles"
  },
  "common": {
    "set_azimuth": 232,
    "fov_left": 32,
    "fov_right": 10,
    "default_percentage": 60,
    "sunset_position": null,
    "sunset_offset": 0,
    "sunrise_offset": 0,
    "max_position": 100,
    "min_position": 0,
    "enable_max_position": false,
    "enable_min_position": false,
    "blind_spot": false,
    "blind_spot_left": 0,
    "blind_spot_right": 0,
    "blind_spot_elevation": 0,
    "min_elevation": null,
    "max_elevation": null
  },
  "vertical": {
    "distance_shaded_area": 1.1,
    "window_height": 2.0,
    "window_depth": 0.0,
    "sill_height": 0.0
  },
  "horizontal": {
    "length_awning": 2.0,
    "angle": 0
  },
  "tilt": {
    "slat_distance": 4.0,
    "slat_depth": 6.0,
    "tilt_mode": "mode1"
  }
}
"""

# Simulation date range
START_DATE = "today"  # "today" or "YYYY-MM-DD"
NUM_DAYS = 1          # 1 = single 24-hour day; increase for multi-day range

CONFIG = json.loads(CONFIG_JSON)
print(f"Loaded config for: {CONFIG['name']} ({CONFIG['cover_type']})")
print(f"Location: {CONFIG['location']['latitude']:.4f}, {CONFIG['location']['longitude']:.4f} — {CONFIG['location']['timezone']}")

In [ ]:
# Cell 3 — Mocks, helpers, and SimulationSunData

loc = CONFIG["location"]
timezone = loc["timezone"]


class MockedHass:
    """Minimal Home Assistant mock using location from exported config."""

    class MockedConfig:
        pass

    config = MockedConfig()
    data = {}


MockedHass.config.latitude = loc["latitude"]
MockedHass.config.longitude = loc["longitude"]
MockedHass.config.elevation = loc["elevation"]
MockedHass.config.time_zone = timezone


class MockedLogger:
    """No-op logger."""

    def debug(self, msg, *args, **kwargs): pass
    def info(self, msg, *args, **kwargs): pass
    def warning(self, msg, *args, **kwargs): pass
    def error(self, msg, *args, **kwargs): pass
    config_name = "Simulation"


mocked_hass = MockedHass()
mocked_logger = MockedLogger()


class SimulationSunData(SunData):
    """SunData subclass that uses a specific date instead of today."""

    def __init__(self, tz, hass, target_date):
        super().__init__(tz, hass)
        self._target_date = target_date

    @property
    def times(self):
        end = self._target_date + timedelta(days=1)
        return pd.date_range(
            start=self._target_date, end=end, freq="5min",
            tz=self.timezone, name="time"
        )

    def sunset(self):
        return self.location.sunset(self._target_date, local=False)

    def sunrise(self):
        return self.location.sunrise(self._target_date, local=False)


# Resolve start date
start_date = date.today() if START_DATE == "today" else date.fromisoformat(START_DATE)
date_range = [start_date + timedelta(days=i) for i in range(NUM_DAYS)]
print(f"Simulating {NUM_DAYS} day(s) starting {start_date}")

In [ ]:
# Cell 4 — Compute cover positions across the date range

c = CONFIG["common"]
v = CONFIG["vertical"]
h = CONFIG["horizontal"]
t = CONFIG["tilt"]
cover_type = CONFIG["cover_type"]

# Common kwargs shared by all cover types
common_kwargs = dict(
    hass=mocked_hass,
    logger=mocked_logger,
    timezone=timezone,
    win_azi=c["set_azimuth"],
    fov_left=c["fov_left"],
    fov_right=c["fov_right"],
    h_def=c["default_percentage"],
    sunset_pos=c["sunset_position"],
    sunset_off=c["sunset_offset"],
    sunrise_off=c["sunrise_offset"],
    max_pos=c["max_position"],
    min_pos=c["min_position"],
    max_pos_bool=c["enable_max_position"],
    min_pos_bool=c["enable_min_position"],
    blind_spot_on=c["blind_spot"],
    blind_spot_left=c["blind_spot_left"],
    blind_spot_right=c["blind_spot_right"],
    blind_spot_elevation=c["blind_spot_elevation"],
    min_elevation=c["min_elevation"],
    max_elevation=c["max_elevation"],
)


def make_cover(row, sim_sun, **extra):
    """Instantiate the correct cover class for a single time step."""
    kwargs = {**common_kwargs, "sol_azi": row["azimuth"], "sol_elev": row["elevation"], **extra}

    if cover_type == "cover_blind":
        cover = AdaptiveVerticalCover(
            **kwargs,
            distance=v["distance_shaded_area"],
            h_win=v["window_height"],
            window_depth=v["window_depth"],
            sill_height=v["sill_height"],
        )
    elif cover_type == "cover_awning":
        cover = AdaptiveHorizontalCover(
            **kwargs,
            distance=v["distance_shaded_area"],
            h_win=v["window_height"],
            window_depth=v["window_depth"],
            sill_height=v["sill_height"],
            awn_length=h["length_awning"],
            awn_angle=h["angle"],
        )
    elif cover_type == "cover_tilt":
        cover = AdaptiveTiltCover(
            **kwargs,
            slat_distance=t["slat_distance"] / 100,
            depth=t["slat_depth"] / 100,
            mode=t["tilt_mode"],
        )
    else:
        raise ValueError(f"Unknown cover_type: {cover_type}")

    cover.sun_data = sim_sun  # Override to use simulation date
    return cover


all_days = []

for sim_date in date_range:
    sim_sun = SimulationSunData(timezone, mocked_hass, sim_date)

    day_df = pd.DataFrame(
        {"azimuth": sim_sun.solar_azimuth, "elevation": sim_sun.solar_elevation}
    )
    day_df.set_index(sim_sun.times, inplace=True)

    positions, valids, gammas = [], [], []
    for _, row in day_df.iterrows():
        cover = make_cover(row, sim_sun)
        positions.append(cover.calculate_percentage())
        valids.append(cover.valid)
        gammas.append(abs(cover.gamma) if cover.gamma is not None else None)

    day_df["position"] = positions
    day_df["sun_in_fov"] = valids
    day_df["gamma"] = gammas
    day_df["_sim_date"] = str(sim_date)
    day_df["_sunrise"] = sim_sun.sunrise()
    day_df["_sunset"] = sim_sun.sunset()

    all_days.append(day_df)

print(f"Computed {NUM_DAYS} day(s) × {len(all_days[0])} time steps each")

In [ ]:
# Cell 5 — Per-day plots
#
# Each day gets its own figure with two subplots:
#   Top:    Sun elevation and azimuth
#   Bottom: Cover position (%) and gamma angle

import os
os.makedirs("outputs", exist_ok=True)

cover_type_label = {"cover_blind": "Vertical Blind", "cover_awning": "Horizontal Awning", "cover_tilt": "Tilt/Venetian"}.get(cover_type, cover_type)

for day_df in all_days:
    sim_date_str = day_df["_sim_date"].iloc[0]
    sunrise_utc = day_df["_sunrise"].iloc[0]
    sunset_utc = day_df["_sunset"].iloc[0]

    # Convert UTC sunrise/sunset to local for annotation
    sunrise_local = pd.Timestamp(sunrise_utc).tz_convert(timezone)
    sunset_local = pd.Timestamp(sunset_utc).tz_convert(timezone)

    in_fov = day_df[day_df["sun_in_fov"]]
    fov_start = in_fov.index[0] if not in_fov.empty else None
    fov_end = in_fov.index[-1] if not in_fov.empty else None

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
    fig.suptitle(f"{CONFIG['name']} — {sim_date_str} ({cover_type_label})", fontsize=13, fontweight="bold")

    # Top: sun position
    ax1.plot(day_df.index, day_df["elevation"], label="Elevation", color="#f59e0b")
    ax1.plot(day_df.index, day_df["azimuth"], label="Azimuth", color="#6366f1", alpha=0.7)
    ax1.set_ylabel("Degrees (°)")
    ax1.legend(loc="upper left")
    ax1.grid(True, alpha=0.3)

    # Shade FOV region in top plot
    if fov_start and fov_end:
        ax1.axvspan(fov_start, fov_end, alpha=0.08, color="yellow", label="Sun in FOV")

    # Bottom: cover position
    ax2.plot(day_df.index, day_df["position"], label="Cover position", color="#10b981", linewidth=2)
    ax2.set_ylabel("Cover position (%)")
    ax2.set_ylim(-5, 105)
    ax2.grid(True, alpha=0.3)

    # Gamma on secondary axis
    ax2b = ax2.twinx()
    ax2b.plot(day_df.index, day_df["gamma"], label="|Gamma|", color="#94a3b8", linewidth=1, linestyle="--", alpha=0.7)
    ax2b.set_ylabel("|Gamma| (°)", color="#94a3b8")
    ax2b.tick_params(axis="y", labelcolor="#94a3b8")

    # Shade FOV region in bottom plot
    if fov_start and fov_end:
        ax2.axvspan(fov_start, fov_end, alpha=0.08, color="yellow")

    # Annotation lines (shared across both subplots)
    for ax in (ax1, ax2):
        ax.axvline(sunrise_local, color="red", linestyle=":", linewidth=1.5)
        ax.axvline(sunset_local, color="red", linestyle=":", linewidth=1.5)
        if fov_start:
            ax.axvline(fov_start, color="gold", linestyle=":", linewidth=1.5)
        if fov_end:
            ax.axvline(fov_end, color="gold", linestyle=":", linewidth=1.5)

    # Labels on top subplot only
    for ts, label, color in [
        (sunrise_local, "sunrise", "red"),
        (sunset_local, "sunset", "red"),
        (fov_start, "FOV start", "goldenrod"),
        (fov_end, "FOV end", "goldenrod"),
    ]:
        if ts is not None:
            ax1.text(ts, ax1.get_ylim()[1], f" {label}", color=color, fontsize=8,
                     ha="left", va="top", rotation=90)

    # Legends
    lines2 = ax2.get_lines() + ax2b.get_lines()
    labels2 = [l.get_label() for l in lines2]
    ax2.legend(lines2, labels2, loc="upper left")

    ax2.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M", tz=timezone))
    ax2.xaxis.set_major_locator(mdates.HourLocator(interval=2))
    plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45, ha="right")
    ax2.set_xlabel("Time")

    plt.tight_layout()
    out_path = f"outputs/sim_{CONFIG['name'].replace(' ', '_')}_{sim_date_str}.png"
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    print(f"Saved: {out_path}")
    plt.show()

In [ ]:
# Cell 6 — Multi-day overlay plot (only shown when NUM_DAYS > 1)
#
# Overlays all days on a shared 0–24h axis so you can compare how
# cover positions shift across days (e.g., around solstice).

if NUM_DAYS > 1:
    fig, ax = plt.subplots(figsize=(14, 5))
    fig.suptitle(f"{CONFIG['name']} — Cover Position Overlay ({NUM_DAYS} days, {cover_type_label})",
                 fontsize=13, fontweight="bold")

    cmap = plt.get_cmap("viridis", NUM_DAYS)

    for i, day_df in enumerate(all_days):
        sim_date_str = day_df["_sim_date"].iloc[0]
        # Normalize index to hours-since-midnight (0–24)
        hours = [(ts.hour + ts.minute / 60) for ts in day_df.index]
        ax.plot(hours, day_df["position"], color=cmap(i), alpha=0.85, label=sim_date_str, linewidth=1.5)

    ax.set_xlabel("Hour of day")
    ax.set_ylabel("Cover position (%)")
    ax.set_xlim(0, 24)
    ax.set_ylim(-5, 105)
    ax.xaxis.set_major_locator(mticker.MultipleLocator(2))
    ax.xaxis.set_minor_locator(mticker.MultipleLocator(1))
    ax.grid(True, alpha=0.3)
    ax.legend(loc="upper right", fontsize=8, ncol=max(1, NUM_DAYS // 10))

    plt.tight_layout()
    out_path = f"outputs/sim_{CONFIG['name'].replace(' ', '_')}_overlay.png"
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    print(f"Saved: {out_path}")
    plt.show()
else:
    print("Multi-day overlay skipped (NUM_DAYS = 1). Set NUM_DAYS > 1 in Cell 2 to enable.")

In [ ]:
# Cell 7 — Summary table

import warnings
rows = []
for day_df in all_days:
    sim_date_str = day_df["_sim_date"].iloc[0]
    sunrise_utc = day_df["_sunrise"].iloc[0]
    sunset_utc = day_df["_sunset"].iloc[0]
    sunrise_local = pd.Timestamp(sunrise_utc).tz_convert(timezone).strftime("%H:%M")
    sunset_local = pd.Timestamp(sunset_utc).tz_convert(timezone).strftime("%H:%M")

    in_fov = day_df[day_df["sun_in_fov"]]
    hours_in_fov = len(in_fov) * 5 / 60

    active = day_df[day_df["position"].notna()]
    rows.append({
        "Date": sim_date_str,
        "Sunrise": sunrise_local,
        "Sunset": sunset_local,
        "Hours in FOV": f"{hours_in_fov:.1f}h",
        "Avg position (%)": f"{active['position'].mean():.1f}" if not active.empty else "—",
        "Max position (%)": f"{active['position'].max():.1f}" if not active.empty else "—",
    })

summary = pd.DataFrame(rows)
print(f"Summary for {CONFIG['name']} ({cover_type_label})")
print(summary.to_string(index=False))